# 4장 1강: Attention 메커니즘 — 실습문제 풀이

> **풀이 기준**: 업로드된 `jena_climate_2009_2016.csv`의 처음 6행으로 코드를 실제 실행한 결과입니다. 출력은 소수 여섯째 자리(가중치는 여덟째 자리)에서 반올림했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 제공 가중치 WQ·WK·WV와 Softmax 함수를 그대로 사용했습니다.

---

## 실습 준비 (제공 코드)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=6)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
raw = df[["T (degC)", "rh (%)"]].to_numpy(dtype=float)
X_all = raw / np.array([10.0, 100.0])
X = X_all[:3]
X_home = X_all[3:6]

WQ = np.array([[1.0, 0.0], [0.0, 1.0]])
WK = np.array([[1.0, 0.0], [0.0, 0.5]])
WV = np.array([[1.0], [0.0]])

print("읽은 데이터 크기:", df.shape)
print(df[["Date Time", "T (degC)", "rh (%)"]].to_string(index=False))
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 두 열의 결측값 수:", df[["T (degC)", "rh (%)"]].isna().sum().sum())
print("실습 X:\n", X)
print("과제 X_home:\n", X_home)
print("가중치 크기:", WQ.shape, WK.shape, WV.shape)

읽은 데이터 크기: (6, 15)
          Date Time  T (degC)  rh (%)
01.01.2009 00:10:00     -8.02    93.3
01.01.2009 00:20:00     -8.41    93.4
01.01.2009 00:30:00     -8.51    93.9
01.01.2009 00:40:00     -8.31    94.2
01.01.2009 00:50:00     -8.27    94.1
01.01.2009 01:00:00     -8.05    94.4
선택 구간 간격(분): [10.]
선택 두 열의 결측값 수: 0
실습 X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]]
과제 X_home:
 [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]
가중치 크기: (2, 2) (2, 2) (2, 1)


In [2]:
def softmax_rows(scores):
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


print("행별 Softmax 함수 준비 완료")

행별 Softmax 함수 준비 완료


---

## 실습 1. Attention과 Query·Key·Value

### 문제 1-1. 같은 입력에서 Q·K·V 만들기

In [3]:
Q = X @ WQ
K = X @ WK
V = X @ WV

print("X:\n", X, "\nshape:", X.shape)
print("Q:\n", Q, "\nshape:", Q.shape)
print("K:\n", K, "\nshape:", K.shape)
print("V:\n", V, "\nshape:", V.shape)
print("q1 =", Q[0], " k1 =", K[0], " v1 =", V[0])

X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] 
shape: (3, 2)
Q:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] 
shape: (3, 2)
K:
 [[-0.802   0.4665]
 [-0.841   0.467 ]
 [-0.851   0.4695]] 
shape: (3, 2)
V:
 [[-0.802]
 [-0.841]
 [-0.851]] 
shape: (3, 1)
q1 = [-0.802  0.933]  k1 = [-0.802   0.4665]  v1 = [-0.802]


#### 답변

**수식 및 계산 과정 (첫 행 x₁ = [−0.802, 0.933])**

- q₁ = [−0.802 × 1 + 0.933 × 0, −0.802 × 0 + 0.933 × 1] = **[−0.802, 0.933]**
- k₁ = [−0.802 × 1 + 0.933 × 0, −0.802 × 0 + 0.933 × 0.5] = **[−0.802, 0.4665]**
- v₁ = [−0.802 × 1 + 0.933 × 0] = **[−0.802]**

**Q1. Attention과 Self-Attention은 어떤 관계인가요? 이번 계산이 Self-Attention인 이유는 무엇인가요?**

- Attention은 Query와 Key를 비교해 비중을 정하고 Value를 가중합하는 **상위 개념**이고, Self-Attention은 Q·K·V를 **모두 같은 입력에서** 만드는 Attention의 한 종류입니다.
- 이번에는 Q·K·V를 모두 같은 입력 X(세 관측 시점)에서 만들었으므로 Self-Attention입니다. 'Self'는 자기 행 하나만 본다는 뜻이 아니며, 각 행은 세 행 모두를 참고합니다.

**Q2. Q·K·V의 크기와 첫 행의 값은 각각 무엇인가요? Q와 K는 두 값인데 V는 한 값이어도 되나요?**

- Q: (3, 2), q₁ = [−0.802, 0.933] / K: (3, 2), k₁ = [−0.802, 0.4665] / V: (3, 1), v₁ = [−0.802]
- 됩니다. Q와 K는 내적으로 서로 비교해야 하므로 성분 수(d_k = 2)가 같아야 하지만, V는 비중에 따라 합칠 정보라서 크기가 달라도 됩니다. 출력 O의 열 수는 V의 열 수를 따릅니다.

**Q3. Q·K·V는 각각 어떤 역할을 하나요? 이번 가중치 행렬은 학습한 결과인가요? WQ·WK·WV와 이후에 구할 A는 어떻게 다른가요?**

- Q: 정보를 모으는 기준(무엇을 찾는가) / K: Q와 비교되어 점수를 만드는 표현 / V: 그 점수에 따른 비중으로 실제 합쳐지는 정보
- 학습한 결과가 아닙니다. 계산을 따라가기 쉽게 정한 고정값입니다.
- WQ·WK·WV는 입력을 Q·K·V로 바꾸는 **모델 파라미터**(실제로는 학습됨)이고, A는 입력이 들어올 때마다 Q·K 점수에서 **새로 계산되는 반영 비중**입니다. 입력이 바뀌면 A도 바뀝니다.

---

## 실습 2. Scaled Dot-Product Attention

### 문제 2-1. 내적부터 Value 가중합까지 계산하기

In [4]:
d_k = K.shape[1]
scores = Q @ K.T
scaled_scores = scores / np.sqrt(d_k)
weights = softmax_rows(scaled_scores)
output = weights @ V

print("d_k =", d_k)
print("scores (S):\n", np.round(scores, 6), "\nshape:", scores.shape)
print("scaled_scores (Z):\n", np.round(scaled_scores, 6), "\nshape:", scaled_scores.shape)
print("weights (A):\n", np.round(weights, 8), "\nshape:", weights.shape)
print("행별 가중치 합:", weights.sum(axis=1))
print("output (O):\n", np.round(output, 6), "\nshape:", output.shape)

contrib = weights[0] * V[:, 0]
print("첫 Query의 Value별 기여도:", np.round(contrib, 6), " 합:", round(contrib.sum(), 6))

d_k = 2
scores (S):
 [[1.078448 1.110193 1.120546]
 [1.110193 1.143459 1.154204]
 [1.120546 1.154204 1.165062]] 
shape: (3, 3)
scaled_scores (Z):
 [[0.762578 0.785025 0.792345]
 [0.785025 0.808548 0.816145]
 [0.792345 0.816145 0.823823]] 
shape: (3, 3)
weights (A):
 [[0.32755578 0.33499149 0.33745273]
 [0.32728819 0.33507812 0.33763369]
 [0.32721829 0.33509955 0.33768215]] 
shape: (3, 3)
행별 가중치 합: [1. 1. 1.]
output (O):
 [[-0.8316  ]
 [-0.831612]
 [-0.831615]] 
shape: (3, 1)
첫 Query의 Value별 기여도: [-0.2627   -0.281728 -0.287172]  합: -0.8316


#### 답변

**수식 및 계산 과정 (첫 Query)**

- s₁₁ = q₁ · k₁ = (−0.802)(−0.802) + (0.933)(0.4665) = 0.643204 + 0.4352445 = **1.0784485**
- s₁₂ = q₁ · k₂ = (−0.802)(−0.841) + (0.933)(0.467) = 0.674482 + 0.435711 = **1.110193**
- s₁₃ = q₁ · k₃ = (−0.802)(−0.851) + (0.933)(0.4695) = 0.682502 + 0.4380435 = **1.1205455**
- √2로 나누기: [0.762578, 0.785025, 0.792345]
- Softmax: [0.32755578, 0.33499149, 0.33745273] (합 1)
- 출력: 0.32755578 × (−0.802) + 0.33499149 × (−0.841) + 0.33745273 × (−0.851) ≈ −0.262700 − 0.281728 − 0.287172 = **−0.831600**

**Q1. 첫 Query와 세 Key의 내적 점수는 각각 얼마인가요?**

- **1.0784485, 1.110193, 1.1205455**입니다. 첫 내적은 (−0.802)(−0.802) + (0.933)(0.4665) = 0.643204 + 0.4352445 = 1.0784485입니다.

**Q2. 첫 Query의 Attention 가중치 3개와 그 합은 얼마인가요? 세 행렬 S·Z·A의 크기와 최종 O의 크기는 무엇인가요?**

- 가중치는 **[0.32755578, 0.33499149, 0.33745273]**이고 합은 **1**입니다.
- S·Z·A는 모두 **(3, 3)**(Query 3개 × Key 3개), O는 **(3, 1)**(Query마다 출력 1개)입니다.

**Q3. 첫 Query의 가중합 결과는 얼마인가요? Attention 가중치는 양수인데 출력이 음수일 수 있는 이유는 무엇인가요?**

- 약 **−0.831600**입니다.
- 가중치는 양수지만, 합쳐지는 Value(기온/10)가 모두 음수(−0.802, −0.841, −0.851)이기 때문입니다. 출력은 Value들의 가중평균이므로 Value 범위 안의 음수가 됩니다.

**Q4. 첫 Query가 가장 큰 비중을 주는 관측은 무엇인가요? 그 사실만으로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 있나요?**

- 세 번째 관측 **00:30**(가중치 0.33745273)입니다.
- 결론 내릴 수 없습니다. 이 비중은 고정된 임의 가중치로 계산한 Q·K 유사도일 뿐입니다. 세 비중의 차이도 0.01 정도로 작고, 미래 기온을 예측하거나 학습한 모델도 아니므로 원인 관계를 말할 수 없습니다.

---

### 문제 2-2. 왜 내적 점수를 √d_k로 나눌까요?

In [5]:
weights_no_scale = softmax_rows(scores)

compare = pd.DataFrame({
    "Key": ["k1 (00:10)", "k2 (00:20)", "k3 (00:30)"],
    "크기 조정 있음": np.round(weights[0], 8),
    "크기 조정 없음": np.round(weights_no_scale[0], 8),
})
print(compare.to_string(index=False))
print("첫 행 최댓값  조정 있음:", round(weights[0].max(), 8), " 조정 없음:", round(weights_no_scale[0].max(), 8))
print("첫 행 합      조정 있음:", weights[0].sum(), " 조정 없음:", weights_no_scale[0].sum())

       Key  크기 조정 있음  크기 조정 없음
k1 (00:10)  0.327556  0.325177
k2 (00:20)  0.334991  0.335665
k3 (00:30)  0.337453  0.339158
첫 행 최댓값  조정 있음: 0.33745273  조정 없음: 0.33915807
첫 행 합      조정 있음: 1.0  조정 없음: 0.9999999999999999


#### 답변

**Q1. 이번에는 왜 √3이나 √1이 아니라 √2로 나누나요? 첫 행 가중치의 최댓값은 조정 전후에 각각 얼마인가요?**

- d_k는 **Key 한 행의 성분 수**이고 이번 Key는 [기온, 습도 × 0.5]의 2개 성분이므로 d_k = 2입니다. 3은 관측(행) 수이고, 1은 V의 성분 수이므로 해당하지 않습니다.
- 최댓값은 조정 있음 **0.33745273**, 조정 없음 **0.33915807**입니다.

**Q2. Scaled Dot-Product Attention에서 크기 조정은 왜 하나요? 이번 예시에서 차이가 아주 크다고 말할 수 있나요?**

- d_k가 커지면 내적 값의 크기도 커지기 쉽고, 큰 점수에 Softmax를 적용하면 비중이 한 곳에 거의 몰립니다. 그러면 기울기가 매우 작아져 학습이 어려워지므로, √d_k로 나눠 점수 크기를 적절히 줄입니다.
- 이번에는 d_k = 2로 작고 점수 차이도 작아서, 최댓값 차이가 약 0.0017에 불과합니다. 차이가 아주 크다고 말할 수 없습니다. 다만 조정하면 비중이 조금 더 고르게(평평하게) 된다는 방향은 확인할 수 있습니다.

**Q3. √d_k로 나누는 것만으로 가중치 합이 1이 되나요? 한 Query의 점수가 모두 같다면 Softmax 가중치는 어떻게 되나요?**

- 아닙니다. 나누기는 점수 크기만 바꾸며, 합을 1로 만드는 것은 **Softmax**입니다. 조정 전후 모두 Softmax를 거쳤기 때문에 합이 1입니다.
- 점수가 모두 같으면 exp 값도 모두 같아서 가중치는 **1/3씩 균등**하게 됩니다.

---

## 기초 과제. 다음 관측 구간에 Attention 적용하기

### 문제 3-1. 동일한 계산으로 다른 구간 요약하기

In [6]:
Q_home = X_home @ WQ
K_home = X_home @ WK
V_home = X_home @ WV

d_home = K_home.shape[1]
scores_home = Q_home @ K_home.T
scaled_home = scores_home / np.sqrt(d_home)
weights_home = softmax_rows(scaled_home)
output_home = weights_home @ V_home

print("Q_home:\n", Q_home, " shape:", Q_home.shape)
print("K_home:\n", K_home, " shape:", K_home.shape)
print("V_home:\n", V_home, " shape:", V_home.shape)
print("scores_home:\n", np.round(scores_home, 6))
print("weights_home:\n", np.round(weights_home, 8), " shape:", weights_home.shape)
print("행별 가중치 합:", weights_home.sum(axis=1))
print("output_home:\n", np.round(output_home, 6), " shape:", output_home.shape)

home_times = df["Date Time"].iloc[3:6].to_list()
best = int(np.argmax(weights_home[0]))
print("첫 Query(00:40)가 가장 큰 비중을 둔 관측:", home_times[best], " 비중:", round(weights_home[0, best], 8))

Q_home:
 [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]  shape: (3, 2)
K_home:
 [[-0.831   0.471 ]
 [-0.827   0.4705]
 [-0.805   0.472 ]]  shape: (3, 2)
V_home:
 [[-0.831]
 [-0.827]
 [-0.805]]  shape: (3, 1)
scores_home:
 [[1.134243 1.130448 1.113579]
 [1.130448 1.126669 1.109887]
 [1.113579 1.109887 1.093593]]
weights_home:
 [[0.33525381 0.33435537 0.33039081]
 [0.33524444 0.33434993 0.33040564]
 [0.33519254 0.33431862 0.33048884]]  shape: (3, 3)
행별 가중치 합: [1. 1. 1.]
output_home:
 [[-0.821072]
 [-0.821072]
 [-0.82107 ]]  shape: (3, 1)
첫 Query(00:40)가 가장 큰 비중을 둔 관측: 01.01.2009 00:40:00  비중: 0.33525381


#### 답변

**수식 및 계산 과정 (첫 Query, 00:40)**

- 첫 내적: q₁ · k₁ = (−0.831)(−0.831) + (0.942)(0.471) = 0.690561 + 0.443682 = **1.134243**
- 첫 출력: 0.33525381 × (−0.831) + 0.33435537 × (−0.827) + 0.33039081 × (−0.805) ≈ −0.278596 − 0.276512 − 0.265965 = **−0.821072**

**Q1. Q_home·K_home·V_home·weights_home·output_home의 크기는 각각 무엇인가요? 실습과 같은 크기가 되는 이유는 무엇인가요?**

- Q_home (3, 2), K_home (3, 2), V_home (3, 1), weights_home (3, 3), output_home (3, 1)입니다.
- 입력이 같은 모양(관측 3개 × 특성 2개)이고 같은 가중치 행렬을 쓰기 때문입니다. 크기는 값이 아니라 행·열 개수로 정해집니다.

**Q2. 첫 내적 점수와 첫 가중합 출력은 각각 얼마인가요?**

- 첫 내적 점수 **1.134243**, 첫 가중합 출력 약 **−0.821072**입니다(위 대입 식).

**Q3. 첫 Query의 세 가중치와 가장 큰 비중의 시각은 무엇인가요? 전체 가중치의 각 행 합은 얼마인가요?**

- 가중치 **[0.33525381, 0.33435537, 0.33039081]**이며, 가장 큰 비중은 첫 번째 관측 **00:40**입니다.
- 세 행 모두 합이 **1**입니다.

**Q4. 결과 숫자가 실습과 달라진 것은 가중치를 새로 학습했기 때문인가요? 이 출력을 미래 기온 예측이라고 설명해도 되나요?**

- 아닙니다. WQ·WK·WV와 Softmax는 그대로이고, **입력 관측값이 바뀌어서** Q·K·V와 점수, 비중이 달라졌습니다.
- 안 됩니다. 출력은 이미 관측한 세 시점의 Value를 비중으로 섞은 요약값일 뿐, 미래 시점을 예측하도록 학습한 결과가 아닙니다.

---

## 실습 마무리

**1. Attention과 Self-Attention은 각각 무엇인가요? 이번 계산이 Self-Attention인 이유와, Self가 자기 행만 참고한다는 뜻인지 설명하세요.**

- Attention: Query와 Key를 비교해 비중을 구하고, 그 비중으로 Value를 가중합하는 방식입니다.
- Self-Attention: Q·K·V를 모두 **같은 입력 묶음**에서 만드는 Attention입니다.
- 이번에는 같은 X에서 Q·K·V를 모두 만들었으므로 Self-Attention입니다. 'Self'는 같은 입력에서 가져온다는 뜻이며, 각 Query는 자기 행을 포함한 세 행을 모두 참고합니다.

**2. Q·K·V의 역할과 첫 행의 값, 전체 행렬 크기를 쓰세요. WQ·WK·WV와 Attention 가중치 A의 차이도 설명하세요.**

- Q(기준): [−0.802, 0.933], (3, 2) / K(비교 대상): [−0.802, 0.4665], (3, 2) / V(합칠 정보): [−0.802], (3, 1)
- WQ·WK·WV는 Q·K·V를 만드는 **변환 파라미터**(실제 모델에서는 학습됨)이고, A는 현재 입력의 Q·K 점수에서 **매번 계산되는 비중**입니다.

**3. 첫 Query와 첫 Key의 내적을 곱셈부터 계산하세요. Q@K.T에서 전치가 필요한 이유와 √2로 나누는 이유를 각각 설명하세요.**

- (−0.802) × (−0.802) = 0.643204, 0.933 × 0.4665 = 0.4352445 → 합 **1.0784485**
- 전치: Q(3, 2)와 K(3, 2)는 그대로 곱할 수 없습니다. K를 (2, 3)으로 전치하면 (3, 2) @ (2, 3) = (3, 3)이 되어, 모든 Query·Key 쌍의 내적을 한 번에 구할 수 있습니다.
- √2: d_k = 2이므로, 내적 크기가 커져 Softmax가 한쪽으로 쏠리는 것을 막기 위해 √d_k로 나눕니다.

**4. Softmax는 점수를 어떤 순서로 비중으로 바꾸나요? axis=1과 keepdims=True의 역할, 행별 합, 한 행의 점수가 모두 같을 때의 결과를 설명하세요.**

- 순서: (행별 최댓값 빼기) → exp로 양수로 바꾸기 → 같은 행의 exp 합으로 나누기
- axis=1: 각 행(Query) 안에서 열 방향으로 최댓값·합을 구합니다.
- keepdims=True: 결과를 (3, 1) 모양으로 유지해 원래 (3, 3) 행렬의 각 행에 맞춰 빼고 나눌 수 있게 합니다.
- 행별 합은 항상 1이며, 한 행의 점수가 모두 같으면 각 비중은 1/3입니다.

**5. 첫 Query의 비중 [0.32755578, 0.33499149, 0.33745273]을 Value [−0.802, −0.841, −0.851]에 적용해 출력을 구하세요. 양수 비중에서 음수 출력이 나오는 이유와 출력 세 개의 의미도 설명하세요.**

- 0.32755578 × (−0.802) = −0.262700, 0.33499149 × (−0.841) = −0.281728, 0.33745273 × (−0.851) = −0.287172 → 합 약 **−0.831600**
- 비중은 양수지만 Value가 모두 음수라 가중평균도 음수입니다.
- 출력 세 개(−0.831600, −0.831612, −0.831615)는 각 Query 시점(00:10, 00:20, 00:30)이 세 관측의 Value를 자기 기준의 비중으로 섞어 만든 **요약 표현**입니다.

**6. 크기 조정 전후 첫 행의 최대 비중은 어떻게 달라지나요? 과제 구간에서 가장 큰 비중의 시각과 첫 출력을 쓰고, 이 결과가 학습·미래 예측·인과관계를 뜻하는지 설명하세요.**

- 첫 행 최대 비중은 조정 있음 0.33745273, 조정 없음 0.33915807로, 조정하면 아주 조금 더 고르게 됩니다.
- 과제 구간의 첫 Query는 **00:40**에 가장 큰 비중(0.33525381)을 두었고, 첫 출력은 약 **−0.821072**입니다.
- 학습·미래 예측·인과관계 어느 것도 뜻하지 않습니다. 고정 가중치로 이미 관측한 값의 유사도 비중을 계산하고 가중평균한 결과일 뿐입니다.